In [36]:
import pandas as pd
import numpy as np
import os
import requests
from datetime import datetime
from pathlib import Path


In [37]:
# =============================================================================
# ── 4. SILVER LAYER: Aggregation & Transformation (Commune level + Unpivot)
# =============================================================================
print("\n⚙️ Processing BRONZE -> SILVER...")

# Load Bronze
path_rhone_bronze = '../../../data/bronze/2022-pres-t2-commune-rhone-69-bronze.csv'

df_b = pd.read_csv(path_rhone_bronze, sep=";", dtype=str)

# --- A. Aggregate the Base Commune Metrics ---
base_cols = ['Code du département', 'Libellé du département', 'Code de la commune', 'Libellé de la commune']
metrics_cols = ['Inscrits', 'Abstentions', 'Votants', 'Blancs', 'Nuls', 'Exprimés']

# Convert metrics to integer for summing
for c in metrics_cols:
    df_b[c] = df_b[c].astype(int)




⚙️ Processing BRONZE -> SILVER...


In [38]:
# Group by Commune and sum the metrics
df_commune = df_b.groupby(base_cols)[metrics_cols].sum().reset_index()

# Recalculate percentages correctly at the commune level
df_commune['% Abs/Ins'] = (df_commune['Abstentions'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Vot/Ins'] = (df_commune['Votants'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Blancs/Ins'] = (df_commune['Blancs'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Blancs/Vot'] = np.where(df_commune['Votants'] > 0, (df_commune['Blancs'] / df_commune['Votants'] * 100), 0).round(2)
df_commune['% Nuls/Ins'] = (df_commune['Nuls'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Nuls/Vot'] = np.where(df_commune['Votants'] > 0, (df_commune['Nuls'] / df_commune['Votants'] * 100), 0).round(2)
df_commune['% Exp/Ins'] = (df_commune['Exprimés'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Exp/Vot'] = np.where(df_commune['Votants'] > 0, (df_commune['Exprimés'] / df_commune['Votants'] * 100), 0).round(2)



In [39]:
# --- B. Unpivot / Melt the Candidates Data ---
cand_dfs = []
for i in range(1, 3):
    # Extract columns for this specific candidate
    cand_subset = base_cols + [f'N°Panneau_{i}', f'Sexe_{i}', f'Nom_{i}', f'Prénom_{i}', f'Voix_{i}']
    temp = df_b[cand_subset].copy()
    
    # Standardize column names
    temp.rename(columns={
        f'N°Panneau_{i}': 'N°Panneau',
        f'Sexe_{i}': 'Sexe',
        f'Nom_{i}': 'Nom', 
        f'Prénom_{i}': 'Prénom',
        f'Voix_{i}': 'Voix'
    }, inplace=True)
    cand_dfs.append(temp)

In [40]:
# Combine all candidates vertically
df_cands = pd.concat(cand_dfs, ignore_index=True)
df_cands['Voix'] = df_cands['Voix'].astype(int)

# Group by Commune AND Candidate to sum their votes
cand_group_cols = base_cols + ['N°Panneau', 'Sexe', 'Nom', 'Prénom']
df_cands_grouped = df_cands.groupby(cand_group_cols)['Voix'].sum().reset_index()

# --- C. Merge & Finalize Silver Dataset ---
# Join candidate votes with commune total metrics
df_silver = pd.merge(df_cands_grouped, df_commune, on=base_cols, how='left')

# Recalculate Candidate Percentages
df_silver['% Voix/Ins'] = (df_silver['Voix'] / df_silver['Inscrits'] * 100).round(2)
df_silver['% Voix/Exp'] = np.where(df_silver['Exprimés'] > 0, (df_silver['Voix'] / df_silver['Exprimés'] * 100), 0).round(2)



In [41]:
# Reorder columns for readability
final_cols = base_cols + metrics_cols + [
    '% Abs/Ins', '% Vot/Ins', '% Blancs/Ins', '% Blancs/Vot',
    '% Nuls/Ins', '% Nuls/Vot', '% Exp/Ins', '% Exp/Vot',
    'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp'
]
df_silver = df_silver[final_cols]



In [42]:
def add_election_metadata(df, annee_election, tour):
    df = df.copy()

    # Ensure clean types
    df["annee_election"] = pd.to_numeric(annee_election, errors="coerce")
    df["tour"] = pd.to_numeric(tour, errors="coerce")

    return df

df_silver = add_election_metadata(df_silver, 2022, 2)

print(f"Colonnes finales: {list(df_silver.columns)}")
display(df_silver.head(5))

Colonnes finales: ['Code du département', 'Libellé du département', 'Code de la commune', 'Libellé de la commune', 'Inscrits', 'Abstentions', 'Votants', 'Blancs', 'Nuls', 'Exprimés', '% Abs/Ins', '% Vot/Ins', '% Blancs/Ins', '% Blancs/Vot', '% Nuls/Ins', '% Nuls/Vot', '% Exp/Ins', '% Exp/Vot', 'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp', 'annee_election', 'tour']


,Code du département,Libellé du département,Code de la commune,Libellé de la commune,Inscrits,Abstentions,Votants,Blancs,Nuls,Exprimés,...,% Exp/Vot,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp,annee_election,tour
0,69,Rhône,001,Affoux,316,49,267,21,9,237,...,88.76,1,M,MACRON,Emmanuel,98,31.01,41.35,2022,2
1,69,Rhône,001,Affoux,316,49,267,21,9,237,...,88.76,2,F,LE PEN,Marine,139,43.99,58.65,2022,2
2,69,Rhône,002,Aigueperse,217,47,170,9,4,157,...,92.35,1,M,MACRON,Emmanuel,74,34.10,47.13,2022,2
3,69,Rhône,002,Aigueperse,217,47,170,9,4,157,...,92.35,2,F,LE PEN,Marine,83,38.25,52.87,2022,2
4,69,Rhône,003,Albigny-sur-Saône,1860,504,1356,81,18,1257,...,92.70,1,M,MACRON,Emmanuel,823,44.25,65.47,2022,2


In [43]:
def add_nuance_to_silver_data(df_silver, reference_file_path):
    import pandas as pd

    df_ref = pd.read_csv(reference_file_path, encoding="utf-8-sig").rename(columns={
        "Nom": "Prénom_ref",
        "Prénom": "Nom_ref"
    })

    for col in ["Nom", "Prénom"]:
        df_silver[col] = df_silver[col].astype(str).str.strip()

    for col in ["Nom_ref", "Prénom_ref"]:
        df_ref[col] = df_ref[col].astype(str).str.strip()

    df_ref = df_ref.drop_duplicates(subset=["Nom_ref", "Prénom_ref"])

    df_enriched = df_silver.merge(
        df_ref[["Nom_ref", "Prénom_ref", "nuance"]],
        left_on=["Nom", "Prénom"],
        right_on=["Nom_ref", "Prénom_ref"],
        how="left"
    ).drop(columns=["Nom_ref", "Prénom_ref"])

    return df_enriched

# Ajouter la nuance politique
reference_path = '../../../data/reference/nuance_politique_candidates_master.csv'
df_silver = add_nuance_to_silver_data(df_silver, reference_path)

# Vérifier que la colonne est bien présente
print(f"Colonnes finales: {list(df_silver.columns)}")
display(df_silver.head(5))

path_rhone_silver = '../../../data/silver/2022-pres-t2-commune-rhone-69-silver.csv'
df_silver.to_csv(path_rhone_silver, index=False, sep=";", encoding="utf-8")

print(f"✅ Updated SILVER dataset created: {path_rhone_silver}")
print(f"📊 Columns: {list(df_silver.columns)}")
print(f"📊 Total Rows: {len(df_silver)}")
display(df_silver.head(5))



print(f"✅ SILVER dataset created: {path_rhone_silver}")
print(f"📊 Unique Communes: {df_silver['Code de la commune'].nunique()}")
print(f"📊 Total Rows: {len(df_silver)} (Communes × 11 Candidates)")

# Show a clean preview for a single commune (e.g., 'Affoux')
print("\n🔍 Sample preview (Commune: Affoux):")
display(df_silver[df_silver['Libellé de la commune'] == 'Affoux'].head(5))

Colonnes finales: ['Code du département', 'Libellé du département', 'Code de la commune', 'Libellé de la commune', 'Inscrits', 'Abstentions', 'Votants', 'Blancs', 'Nuls', 'Exprimés', '% Abs/Ins', '% Vot/Ins', '% Blancs/Ins', '% Blancs/Vot', '% Nuls/Ins', '% Nuls/Vot', '% Exp/Ins', '% Exp/Vot', 'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp', 'annee_election', 'tour', 'nuance']


,Code du département,Libellé du département,Code de la commune,Libellé de la commune,Inscrits,Abstentions,Votants,Blancs,Nuls,Exprimés,...,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp,annee_election,tour,nuance
0,69,Rhône,001,Affoux,316,49,267,21,9,237,...,1,M,MACRON,Emmanuel,98,31.01,41.35,2022,2,REM
1,69,Rhône,001,Affoux,316,49,267,21,9,237,...,2,F,LE PEN,Marine,139,43.99,58.65,2022,2,RN
2,69,Rhône,002,Aigueperse,217,47,170,9,4,157,...,1,M,MACRON,Emmanuel,74,34.10,47.13,2022,2,REM
3,69,Rhône,002,Aigueperse,217,47,170,9,4,157,...,2,F,LE PEN,Marine,83,38.25,52.87,2022,2,RN
4,69,Rhône,003,Albigny-sur-Saône,1860,504,1356,81,18,1257,...,1,M,MACRON,Emmanuel,823,44.25,65.47,2022,2,REM


✅ Updated SILVER dataset created: ../../../data/silver/2022-pres-t2-commune-rhone-69-silver.csv
📊 Columns: ['Code du département', 'Libellé du département', 'Code de la commune', 'Libellé de la commune', 'Inscrits', 'Abstentions', 'Votants', 'Blancs', 'Nuls', 'Exprimés', '% Abs/Ins', '% Vot/Ins', '% Blancs/Ins', '% Blancs/Vot', '% Nuls/Ins', '% Nuls/Vot', '% Exp/Ins', '% Exp/Vot', 'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp', 'annee_election', 'tour', 'nuance']
📊 Total Rows: 534


,Code du département,Libellé du département,Code de la commune,Libellé de la commune,Inscrits,Abstentions,Votants,Blancs,Nuls,Exprimés,...,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp,annee_election,tour,nuance
0,69,Rhône,001,Affoux,316,49,267,21,9,237,...,1,M,MACRON,Emmanuel,98,31.01,41.35,2022,2,REM
1,69,Rhône,001,Affoux,316,49,267,21,9,237,...,2,F,LE PEN,Marine,139,43.99,58.65,2022,2,RN
2,69,Rhône,002,Aigueperse,217,47,170,9,4,157,...,1,M,MACRON,Emmanuel,74,34.10,47.13,2022,2,REM
3,69,Rhône,002,Aigueperse,217,47,170,9,4,157,...,2,F,LE PEN,Marine,83,38.25,52.87,2022,2,RN
4,69,Rhône,003,Albigny-sur-Saône,1860,504,1356,81,18,1257,...,1,M,MACRON,Emmanuel,823,44.25,65.47,2022,2,REM


✅ SILVER dataset created: ../../../data/silver/2022-pres-t2-commune-rhone-69-silver.csv
📊 Unique Communes: 267
📊 Total Rows: 534 (Communes × 11 Candidates)

🔍 Sample preview (Commune: Affoux):


,Code du département,Libellé du département,Code de la commune,Libellé de la commune,Inscrits,Abstentions,Votants,Blancs,Nuls,Exprimés,...,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp,annee_election,tour,nuance
0,69,Rhône,001,Affoux,316,49,267,21,9,237,...,1,M,MACRON,Emmanuel,98,31.01,41.35,2022,2,REM
1,69,Rhône,001,Affoux,316,49,267,21,9,237,...,2,F,LE PEN,Marine,139,43.99,58.65,2022,2,RN


In [44]:
import pandas as pd
from datetime import datetime

def standardize_common_columns(
    df,
    source_dataset,
    date_refresh=None
):
    df = df.copy()

    rename_map = {
        "Code du département": "code_departement",
        "Libellé du département": "libelle_departement",
        "Code de la commune": "code_commune",
        "Libellé de la commune": "libelle_commune",
        "Inscrits": "inscrits",
        "Abstentions": "abstentions",
        "Votants": "votants",
        "Blancs": "blancs",
        "Nuls": "nuls",
        "Exprimés": "exprimes",
        "% Abs/Ins": "pct_abs_ins",
        "% Vot/Ins": "pct_vot_ins",
        "% Blancs/Ins": "pct_blancs_ins",
        "% Blancs/Vot": "pct_blancs_vot",
        "% Nuls/Ins": "pct_nuls_ins",
        "% Nuls/Vot": "pct_nuls_vot",
        "% Exp/Ins": "pct_exprimes_ins",
        "% Exp/Vot": "pct_exprimes_vot",
        "N°Panneau": "numero_panneau",
        "Sexe": "sexe",
        "Nom": "nom",
        "Prénom": "prenom",
        "Voix": "voix",
        "% Voix/Ins": "pct_voix_ins",
        "% Voix/Exp": "pct_voix_exprimes",
        "annee_election": "annee_election",
        "tour": "tour",
        "nuance": "nuance"
    }

    df = df.rename(columns=rename_map)

    df["code_departement"] = (
        df["code_departement"]
        .astype(str)
        .str.strip()
        .str.replace(".0", "", regex=False)
        .str.zfill(2)
    )

    df["code_commune"] = (
        df["code_commune"]
        .astype(str)
        .str.strip()
        .str.replace(".0", "", regex=False)
        .str.zfill(3)
    )

    df["libelle_departement"] = df["libelle_departement"].astype(str).str.strip()
    df["libelle_commune"] = df["libelle_commune"].astype(str).str.strip()
    df["nom"] = df["nom"].astype(str).str.strip()
    df["prenom"] = df["prenom"].astype(str).str.strip()
    df["nuance"] = df["nuance"].astype(str).str.strip()

    df["annee_election"] = pd.to_numeric(df["annee_election"], errors="coerce").astype("Int64")
    df["tour"] = pd.to_numeric(df["tour"], errors="coerce").astype("Int64")
    df["numero_panneau"] = pd.to_numeric(df["numero_panneau"], errors="coerce").astype("Int64")

    int_cols = [
        "inscrits", "abstentions", "votants", "blancs",
        "nuls", "exprimes", "voix"
    ]
    for col in int_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    float_cols = [
        "pct_abs_ins", "pct_vot_ins", "pct_blancs_ins", "pct_blancs_vot",
        "pct_nuls_ins", "pct_nuls_vot", "pct_exprimes_ins", "pct_exprimes_vot",
        "pct_voix_ins", "pct_voix_exprimes"
    ]
    for col in float_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    

    df["source_dataset"] = source_dataset
    df["date_refresh"] = date_refresh or datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
    ordered_cols = [
        "code_departement",
        "libelle_departement",
        "code_commune",
        "libelle_commune",
        "annee_election",
        "tour",
        "inscrits",
        "abstentions",
        "votants",
        "blancs",
        "nuls",
        "exprimes",
        "pct_abs_ins",
        "pct_vot_ins",
        "pct_blancs_ins",
        "pct_blancs_vot",
        "pct_nuls_ins",
        "pct_nuls_vot",
        "pct_exprimes_ins",
        "pct_exprimes_vot",
        "numero_panneau",
        "sexe",
        "nom",
        "prenom",
        "voix",
        "pct_voix_ins",
        "pct_voix_exprimes",
        "nuance",
        "source_dataset",
        "date_refresh"
    ]
    return df[ordered_cols]


df_silver = standardize_common_columns(
    df_silver,
    source_dataset="2022-pres-t2-commune-rhone-69-silver.csv"
)

print("Colonnes standardisées :", df_silver.columns.tolist())
display(df_silver.head())

path_rhone_silver = '../../../data/silver/2022-pres-t2-commune-rhone-69-silver.csv'
df_silver.to_csv(path_rhone_silver, index=False, sep=";", encoding="utf-8")

Colonnes standardisées : ['code_departement', 'libelle_departement', 'code_commune', 'libelle_commune', 'annee_election', 'tour', 'inscrits', 'abstentions', 'votants', 'blancs', 'nuls', 'exprimes', 'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins', 'pct_blancs_vot', 'pct_nuls_ins', 'pct_nuls_vot', 'pct_exprimes_ins', 'pct_exprimes_vot', 'numero_panneau', 'sexe', 'nom', 'prenom', 'voix', 'pct_voix_ins', 'pct_voix_exprimes', 'nuance', 'source_dataset', 'date_refresh']


,code_departement,libelle_departement,code_commune,libelle_commune,annee_election,tour,inscrits,abstentions,votants,blancs,...,numero_panneau,sexe,nom,prenom,voix,pct_voix_ins,pct_voix_exprimes,nuance,source_dataset,date_refresh
0,69,Rhône,001,Affoux,2022,2,316,49,267,21,...,1,M,MACRON,Emmanuel,98,31.01,41.35,REM,2022-pres-t2-commune-rhone-69-silver.csv,2026-03-24T14:42:55
1,69,Rhône,001,Affoux,2022,2,316,49,267,21,...,2,F,LE PEN,Marine,139,43.99,58.65,RN,2022-pres-t2-commune-rhone-69-silver.csv,2026-03-24T14:42:55
2,69,Rhône,002,Aigueperse,2022,2,217,47,170,9,...,1,M,MACRON,Emmanuel,74,34.10,47.13,REM,2022-pres-t2-commune-rhone-69-silver.csv,2026-03-24T14:42:55
3,69,Rhône,002,Aigueperse,2022,2,217,47,170,9,...,2,F,LE PEN,Marine,83,38.25,52.87,RN,2022-pres-t2-commune-rhone-69-silver.csv,2026-03-24T14:42:55
4,69,Rhône,003,Albigny-sur-Saône,2022,2,1860,504,1356,81,...,1,M,MACRON,Emmanuel,823,44.25,65.47,REM,2022-pres-t2-commune-rhone-69-silver.csv,2026-03-24T14:42:55
